In [1]:
# Import libraries and define configs
import json
import random
import warnings
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import resample

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "figure.dpi": 120,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
    }
)

RANDOM_STATE = 121
N_JOBS = -1
CV_FOLDS = 5
OPERATING_THRESHOLD = 0.065424
OUTREACH_CAPACITIES = (0.01, 0.05, 0.10, 0.20)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

current_directory = Path.cwd().resolve()
project_root_candidates = [current_directory, *current_directory.parents]

PROJECT_ROOT = next(
    (
        candidate_directory
        for candidate_directory in project_root_candidates
        if (candidate_directory / "data" / "processed").exists()
        and (candidate_directory / "notebooks").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/processed and notebooks."
    )

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELING_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "modeling"
PRIVATE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "private"
INTERPRETABILITY_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "interpretability"

INTERPRETABILITY_TABLES_DIR = INTERPRETABILITY_OUTPUT_DIR / "tables"
INTERPRETABILITY_FIGURES_DIR = INTERPRETABILITY_OUTPUT_DIR / "figures"

INTERPRETABILITY_TABLES_DIR.mkdir(parents=True, exist_ok=True)
INTERPRETABILITY_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DONOR_FEATURES_PARQUET_PATH = PROCESSED_DATA_DIR / "donor_features.parquet"
DONOR_FEATURES_CSV_PATH = PROCESSED_DATA_DIR / "donor_features.csv"
FEATURE_DICTIONARY_PATH = REPORTS_DIR / "feature_dictionary.csv"

PRIMARY_PIPELINE_PATH = MODELS_DIR / "final_primary_pipeline.joblib"
BENCHMARK_PIPELINE_PATH = (
    MODELS_DIR / "historical_donor_status_benchmark_pipeline.joblib"
)

FINAL_MODEL_CONFIGURATION_PATH = (
    MODELING_OUTPUT_DIR / "final_model_configuration.json"
)
FINAL_FEATURE_LISTS_PATH = MODELING_OUTPUT_DIR / "final_feature_lists.json"
FINAL_TEST_PREDICTIONS_PATH = (
    PRIVATE_OUTPUT_DIR / "primary_donor_predictions_final_test.csv"
)
PRIMARY_TEST_METRICS_PATH = (
    MODELING_OUTPUT_DIR / "primary_final_test_metrics.csv"
)
PRIMARY_OUTREACH_RESULTS_PATH = (
    MODELING_OUTPUT_DIR / "primary_final_test_outreach_results.csv"
)
BENCHMARK_TEST_METRICS_PATH = (
    MODELING_OUTPUT_DIR / "benchmark_test_metrics.csv"
)
MODELING_ARTIFACT_MANIFEST_PATH = (
    MODELING_OUTPUT_DIR / "modeling_artifact_manifest.csv"
)